Comp 3610 Assignment 4
Part 1 - Experiment Tracking with MLflow

In [2]:
import numpy as np
import pandas as pd
import math, os, time
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

import mlflow
import mlflow.sklearn

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

DATA_PATH = 'cleaned_taxi_data.parquet'
ZONE_PATH = 'taxi_zone_lookup.csv'

df_raw = pd.read_parquet(DATA_PATH)
zone_df = pd.read_csv(ZONE_PATH)
loc_to_borough = zone_df.set_index('LocationID')['Borough'].to_dict()

df = df_raw[df_raw['payment_type'] == 1].copy()

for col in ['tpep_pickup_datetime', 'tpep_dropoff_datetime']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

print(f"Rows after credit card filter: {len(df):,}")


Rows after credit card filter: 2,241,617


In [4]:
def engineering_features(df, loc_to_borough):
    df = df.copy()
    df['pickup_hour']           = df['tpep_pickup_datetime'].dt.hour
    df['pickup_day_of_week']    = df['tpep_pickup_datetime'].dt.dayofweek
    df['is_weekend']            = (df['pickup_day_of_week'] >= 5).astype(int)
    df['trip_duration_minutes'] = (
        (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60
    )
    df['trip_speed_mph'] = np.where(
        df['trip_duration_minutes'] > 0,
        df['trip_distance'] / (df['trip_duration_minutes'] / 60), np.nan
    )
    df['log_trip_distance'] = np.log1p(df['trip_distance'])
    df['fare_per_mile']     = np.where(df['trip_distance'] > 0,
                                       df['fare_amount'] / df['trip_distance'], np.nan)
    df['fare_per_minute']   = np.where(df['trip_duration_minutes'] > 0,
                                       df['fare_amount'] / df['trip_duration_minutes'], np.nan)
    le = LabelEncoder()
    df['pickup_borough']     = df['PULocationID'].map(loc_to_borough).fillna('Unknown')
    df['dropoff_borough']    = df['DOLocationID'].map(loc_to_borough).fillna('Unknown')
    df['pickup_borough_enc'] = le.fit_transform(df['pickup_borough'])
    df['dropoff_borough_enc']= le.fit_transform(df['dropoff_borough'])
    return df

df = engineering_features(df, loc_to_borough)

FEATURE_COLS = [
    'pickup_hour', 'pickup_day_of_week', 'is_weekend',
    'trip_duration_minutes', 'trip_speed_mph', 'log_trip_distance',
    'trip_distance', 'passenger_count', 'fare_amount',
    'fare_per_mile', 'fare_per_minute', 'extra', 'mta_tax',
    'tolls_amount', 'improvement_surcharge',
    'pickup_borough_enc', 'dropoff_borough_enc', 'RatecodeID',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

df_model = df[FEATURE_COLS + ['tip_amount']].dropna()
X = df_model[FEATURE_COLS]
y = df_model['tip_amount']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

Train: 1,793,293  |  Test: 448,324


In [5]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("taxi-tip-prediction")
print("MLflow experiment set: taxi-tip-prediction")

2026/04/18 02:39:19 INFO mlflow.tracking.fluent: Experiment with name 'taxi-tip-prediction' does not exist. Creating a new experiment.


MLflow experiment set: taxi-tip-prediction


In [7]:
def log_regression_metrics(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mlflow.log_metric("mae",  mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2",   r2)
    return {"MAE": mae, "RMSE": rmse, "R2": r2}

In [8]:
with mlflow.start_run(run_name="linear-regression"):
    params = {"fit_intercept": True}
    mlflow.log_params(params)
    mlflow.set_tag("model_type", "LinearRegression")
    mlflow.set_tag("dataset_version", "v1")

    lr = LinearRegression(**params)
    lr.fit(X_train_scaled, y_train)
    preds = lr.predict(X_test_scaled)

    metrics_lr = log_regression_metrics(y_test, preds)
    mlflow.sklearn.log_model(lr, "model",
                             registered_model_name="taxi-tip-regressor")

    print("Linear Regression logged.")
    print(f"  MAE={metrics_lr['MAE']:.4f}  RMSE={metrics_lr['RMSE']:.4f}  R2={metrics_lr['R2']:.4f}")

2026/04/18 02:40:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/18 02:40:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'taxi-tip-regressor'.
2026/04/18 02:40:54 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: taxi-tip-regressor, version 1
Created version '1' of model 'taxi-tip-regressor'.


Linear Regression logged.
  MAE=1.2072  RMSE=2.3913  R2=0.6178
🏃 View run linear-regression at: http://localhost:5000/#/experiments/880846622387795464/runs/6744e868d46c470996898011be75650a
🧪 View experiment at: http://localhost:5000/#/experiments/880846622387795464


In [9]:
RF_SAMPLE = 100_000
X_rf = X_train_scaled[:RF_SAMPLE]
y_rf = y_train.iloc[:RF_SAMPLE]

with mlflow.start_run(run_name="random-forest-regressor"):
    params = {"n_estimators": 30, "max_depth": 10, "random_state": 42}
    mlflow.log_params(params)
    mlflow.set_tag("model_type", "RandomForestRegressor")
    mlflow.set_tag("dataset_version", "v1")
    mlflow.set_tag("note", "Trained on 100k sample due to runtime constraints")

    rf = RandomForestRegressor(**params, n_jobs=-1)
    rf.fit(X_rf, y_rf)
    preds_rf = rf.predict(X_test_scaled)

    metrics_rf = log_regression_metrics(y_test, preds_rf)
    mlflow.sklearn.log_model(rf, "model")

    print("Random Forest logged.")
    print(f"  MAE={metrics_rf['MAE']:.4f}  RMSE={metrics_rf['RMSE']:.4f}  R2={metrics_rf['R2']:.4f}")

2026/04/18 02:42:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/18 02:42:18 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Random Forest logged.
  MAE=1.2041  RMSE=2.4198  R2=0.6087
🏃 View run random-forest-regressor at: http://localhost:5000/#/experiments/880846622387795464/runs/6e2f60cbf45d4061a5ba3ed90313ad0a
🧪 View experiment at: http://localhost:5000/#/experiments/880846622387795464
